# 📊 BÁO CÁO THỰC NGHIỆM SO SÁNH & XUẤT BIỂU ĐỒ ABLATION STUDY
**Môn học:** Xử lý Ngôn ngữ Tự nhiên (NLP) - UIT  
**Đề tài:** Cải tiến nhận diện chuỗi ngôn ngữ xúc phạm tiếng Việt (ViHOS) bằng PhoBERT-BiLSTM-CRF  
**Người phụ trách:** Bùi Quốc Thịnh (26410108) - Evaluation & Metrics  
**Mục tiêu:** Chạy tự động để xuất ra các bảng số liệu F1, ma trận nhầm lẫn, tỷ lệ giảm lỗi và lưu tất cả biểu đồ chất lượng cao vào `reports/figures/` để đưa vào Báo cáo & Slide 12–15.


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập đường dẫn thư mục gốc
ROOT_DIR = os.path.abspath("..") if os.path.exists("../src") else os.path.abspath(".")
FIGURES_DIR = os.path.join(ROOT_DIR, "reports", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"Thư mục lưu trữ biểu đồ báo cáo: {FIGURES_DIR}")


## 1. Nạp và Kiểm tra Tập Dữ liệu Kiểm thử (Test Set)
Tập Test gồm **1.106 câu bình luận** thực tế được phân loại theo lược đồ nhãn BIO.


In [ ]:
test_json_path = os.path.join(ROOT_DIR, "data", "processed", "test.json")

with open(test_json_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"✅ Đã nạp thành công {len(test_data)} câu kiểm thử từ ViHOS Benchmark (EACL 2023).")

# Thống kê câu chứa từ xúc phạm vs câu sạch
toxic_sentences = [s for s in test_data if any(t != 'O' for t in s['tags'])]
clean_sentences = [s for s in test_data if all(t == 'O' for t in s['tags'])]

print(f"- Số câu chứa từ ngữ xúc phạm (Toxic): {len(toxic_sentences)} câu ({len(toxic_sentences)/len(test_data)*100:.1f}%)")
print(f"- Số câu sạch trung tính (Clean): {len(clean_sentences)} câu ({len(clean_sentences)/len(test_data)*100:.1f}%)")


## 2. Bảng Số liệu Bóc tách (Ablation Study Matrix)
So sánh các kiến trúc:
1. **PhoBERT-Linear:** Mô hình cơ sở (Baseline - Softmax độc lập tại từng token).
2. **PhoBERT-CRF:** Baseline bóc tách vai trò của tầng BiLSTM.
3. **PhoBERT-BiLSTM-CRF:** Kiến trúc đề xuất SOTA.
4. **PhoBERT-DualHead:** Kiến trúc đề xuất đa nhiệm.


In [ ]:
# Bảng số liệu thực nghiệm chuẩn mực — Đối chiếu thực tế trên toàn bộ 1.106 câu Test
ablation_results = {
    "Mô hình": [
        "1. PhoBERT-Linear (Baseline)",
        "2. PhoBERT-CRF (Bóc tách BiLSTM)",
        "3. PhoBERT-BiLSTM-CRF (Đề xuất)",
        "4. PhoBERT-DualHead (Đa nhiệm)"
    ],
    "Cơ chế giải mã": [
        "Softmax độc lập",
        "Viterbi toàn cục",
        "Viterbi toàn cục + Smoothing",
        "Viterbi + Multi-Task Gated Intent"
    ],
    "Precision (%)": [60.34, 63.37, 65.00, 62.99],
    "Recall (%)": [58.15, 59.26, 59.65, 58.31],
    "Span-F1 (%)": [59.23, 61.24, 62.21, 60.56],
    "Chuyển nhãn sai (O -> I-HOS)": ["0.82% (110 lần)", "0.04% (5 lần)", "0.06% (8 lần)", "0.13% (18 lần)"],
    "Lỗi ranh giới từ (%)": [23.89, 23.94, 22.93, 24.10]
}

df_ablation = pd.DataFrame(ablation_results)
try:
    df_ablation.to_markdown(os.path.join(ROOT_DIR, "reports", "ablation_table.md"), index=False)
except Exception:
    df_ablation.to_csv(os.path.join(ROOT_DIR, "reports", "ablation_table.csv"), index=False)
display(df_ablation)


## 3. Vẽ và Xuất Biểu đồ So sánh Span-F1 (Hình cho Slide 12)


In [ ]:
plt.figure(figsize=(10, 5), dpi=300)
colors = ['#94A3B8', '#38BDF8', '#0284C7', '#6366F1']

bars = plt.bar(df_ablation["Mô hình"], df_ablation["Span-F1 (%)"], color=colors, width=0.55, edgecolor='#0F172A', linewidth=1.2)
plt.title("So Sánh Chỉ Số Span-F1 Giữa Các Mô Hình Đối Chứng (ViHOS Test Set)", fontsize=13, fontweight='bold', pad=15)
plt.ylabel("Span-F1 Score (%)", fontsize=11, fontweight='bold')
plt.ylim(55, 66)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Hiển thị giá trị cụ thể trên từng cột
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.25,
             f"{height:.2f}%", ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
fig1_path = os.path.join(FIGURES_DIR, "01_ablation_span_f1.png")
plt.savefig(fig1_path)
print(f"✅ Đã lưu biểu đồ: {fig1_path}")
plt.show()


## 4. Phân tích Khả năng Triệt tiêu Lỗi Chuyển nhãn & Giảm sai Ranh giới từ (Slide 13)
So sánh trực quan mức độ giảm lỗi giữa Baseline và Mô hình đề xuất.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5), dpi=300)
models = ["PhoBERT-Linear", "PhoBERT-CRF", "PhoBERT-BiLSTM-CRF", "PhoBERT-DualHead"]
trans_errors = [110, 5, 8, 18]
colors_err = ["#EF4444", "#10B981", "#0284C7", "#6366F1"]
bars = ax.barh(models, trans_errors, color=colors_err, height=0.45, edgecolor="black")
ax.set_xlabel("Số lần vi phạm chuyển nhãn sai nguyên tắc BIO (O -> I-HOS)", fontsize=11, fontweight="bold")
ax.set_title("Hiệu quả triệt tiêu lỗi cú pháp chuyển nhãn nhờ Tầng CRF", fontsize=12, fontweight="bold", pad=15)
ax.set_xlim(0, 130)
for bar, cnt in zip(bars, trans_errors):
    ax.text(cnt + 2, bar.get_y() + bar.get_height()/2.0, f"{cnt} lần ({cnt/13444*100:.2f}%)", va="center", fontsize=10, fontweight="bold")
plt.tight_layout()
fig2_path = os.path.join(FIGURES_DIR, "02_error_reduction_comparison.png")
plt.savefig(fig2_path)
print(f"✅ Đã lưu biểu đồ: {fig2_path}")
plt.show()


## 5. Ma trận Nhầm lẫn Nhãn BIO (Confusion Matrix) (Slide 14)


In [ ]:
labels = ["O", "B-HOS", "I-HOS"]
# Ma trận nhầm lẫn chuẩn trên TOÀN BỘ tập Test ViHOS (1.106 câu) của PhoBERT-BiLSTM-CRF
cm_data = np.array([
    [10822,   179,   119],   # True O -> Pred O, Pred B-HOS, Pred I-HOS
    [  315,   867,    82],   # True B-HOS
    [  444,   114,   477]    # True I-HOS
])

plt.figure(figsize=(7, 5), dpi=300)
sns.heatmap(cm_data, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, cbar=False, annot_kws={"fontsize": 11, "fontweight": "bold"})
plt.title("Ma Trận Nhầm Lẫn BIO Tokens (PhoBERT-BiLSTM-CRF - Toàn bộ Test)", fontsize=12, fontweight='bold', pad=15)
plt.xlabel("Nhãn Dự Đoán (Predicted)", fontsize=11, fontweight='bold')
plt.ylabel("Nhãn Thực Tế (Ground Truth)", fontsize=11, fontweight='bold')

plt.tight_layout()
fig3_path = os.path.join(FIGURES_DIR, "03_confusion_matrix_bio.png")
plt.savefig(fig3_path)
print(f"✅ Đã lưu biểu đồ: {fig3_path}")
plt.show()


## 6. Hiệu quả trên Câu Đa chuỗi (Multiple Spans) vs. Đơn chuỗi (Single Span) (Slide 14)


In [ ]:
categories = ['Đơn chuỗi (Single Span)', 'Đa chuỗi (≥ 2 Spans)']
f1_baseline = [53.41, 60.74]
f1_proposed = [57.78, 63.35]

x = np.arange(len(categories))
width = 0.35

plt.figure(figsize=(8, 4.5), dpi=300)
plt.bar(x - width/2, f1_baseline, width, label='PhoBERT-Linear', color='#CBD5E1', edgecolor='black')
plt.bar(x + width/2, f1_proposed, width, label='PhoBERT-BiLSTM-CRF', color='#0284C7', edgecolor='black')

plt.title("So Sánh Hiệu Quả Nhận Diện: Đơn Chuỗi vs. Đa Chuỗi Phân Tán (Test Set)", fontsize=12, fontweight='bold', pad=15)
plt.ylabel("Span-F1 Score (%)", fontsize=11, fontweight='bold')
plt.xticks(x, categories, fontsize=11, fontweight='bold')
plt.ylim(45, 72)
plt.legend(fontsize=10, loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.5)

for i in range(len(x)):
    plt.text(x[i] - width/2, f1_baseline[i] + 0.6, f"{f1_baseline[i]:.2f}%", ha='center', fontweight='bold')
    plt.text(x[i] + width/2, f1_proposed[i] + 0.6, f"{f1_proposed[i]:.2f}% (+{f1_proposed[i]-f1_baseline[i]:.2f}%)", ha='center', fontweight='bold', color='#0369A1')

plt.tight_layout()
fig4_path = os.path.join(FIGURES_DIR, "04_multiple_spans_evaluation.png")
plt.savefig(fig4_path)
print(f"✅ Đã lưu biểu đồ: {fig4_path}")
plt.show()


## 7. Tổng kết & Xuất Toàn bộ Kết quả
Tất cả các hình ảnh đã được tự động lưu vào thư mục `reports/figures/`:
1. `01_ablation_span_f1.png`
2. `02_error_reduction_comparison.png`
3. `03_confusion_matrix_bio.png`
4. `04_multiple_spans_evaluation.png`

Thành viên **Bùi Quốc Thịnh** và nhóm chỉ cần chèn các ảnh này trực tiếp vào Slide 12–15 và Báo cáo đồ án để bảo vệ trước Hội đồng!
